Day 5, Topic 4: np.einsum – Einstein Summation for Matrix Products, Traces, Batches

## 📖 What is `np.einsum`?

**Einstein Summation** is a compact notation invented by Albert Einstein to express tensor contractions without writing explicit loops or sums.

`np.einsum(subscript_string, *operands)` lets you describe almost any linear-algebra operation — element-wise products, dot products, matrix multiplications, traces, outer products, batch contractions — using a single concise string.

### Reading the subscript string

```
'ij, jk -> ik'
```

| Part | Meaning |
|------|---------|
| `ij` | First operand has axes named `i` (rows) and `j` (cols) |
| `jk` | Second operand has axes named `j` and `k` |
| `->` | Separator: left = inputs, right = output |
| `ik` | Output has axes `i` and `k` |
| `j` missing from output | **Sum over** (contract) axis `j` |

### The three rules
1. **Repeated index on the same operand** → diagonal / trace (`'ii->'`)
2. **Repeated index across operands** → contraction / multiplication along that axis
3. **Index absent from output** → summed away

### Why use it?
- Concise and readable once you know the notation
- Can be faster than chained NumPy calls (avoids intermediates) especially with `optimize=True`
- Handles batched operations elegantly that would otherwise need `np.tensordot` + `np.moveaxis` tricks


### Example 1 — Element-wise Multiplication

```
'i, i -> i'
```
- Both inputs have one axis `i`
- Output also has `i` — nothing is summed away
- Result: each element multiplied independently

This is identical to `a * b` but written in einsum notation.


In [1]:
#Example 1: Element‑wise Multiplication
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

#Numpy edition
np_result = a * b

#einsum edition
einsum_result = np.einsum('i,i->i', a, b)

print(einsum_result)

[ 4 10 18]


### Example 2 — Vector Dot Product

```
'i, i -> '   (empty output — scalar result)
```
- Same repeated index `i` as above
- But `i` is **absent from the output** → sum over all `i`
- Result: scalar = Σ aᵢ · bᵢ

Equivalent to `np.dot(a, b)`.


In [2]:
#Example 2: Vector Dot Product
einsum_result = np.einsum('i, i->', a, b)
print(einsum_result)

32


### Example 3 — Matrix Transpose

```
'ij -> ji'
```
- Input has axes `i` (rows), `j` (cols)
- Output swaps them: `j` becomes the first axis, `i` the second
- No contraction — just an axis permutation

Equivalent to `M.T`.  
For higher-dimensional arrays this generalises: `'ijk -> kji'` reverses all three axes.


In [4]:
#Example 3: Matrix Transpose
M = np.arange(12).reshape(4, 3)

#Numpy edition
result = M.T

#einsum edition
einsum_result = np.einsum('ij->ji', M)

print(M)
print(einsum_result)

[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
[[ 0  3  6  9]
 [ 1  4  7 10]
 [ 2  5  8 11]]


### Example 4 — Sum Over Rows / Columns

```
'ij -> j'   # sum over rows → keep columns
'ij -> i'   # sum over columns → keep rows
```

- `j` missing from output in first case → sum axis 0 (rows)
- `i` missing from output in second case → sum axis 1 (columns)

Equivalent to `arr.sum(axis=0)` and `arr.sum(axis=1)`.


In [6]:
#Example 4: Sum Over Rows or Columns
arr = np.arange(6).reshape(2, 3)

col = np.einsum('ij->j', arr)
print(col)

row = np.einsum('ij->i', arr)
print(row)

[3 5 7]
[ 3 12]


### Example 5 — Trace (Sum of Diagonal)

```
'ii -> '
```
- **Same index `i` repeated on the same operand** → select the diagonal elements
- Empty output → sum them into a scalar

Equivalent to `np.trace(M)`.


In [7]:
#Example 5: Trace (Sum of Diagonal)
M = np.arange(9).reshape(3, 3)

trace = np.einsum('ii->', M)
print(trace)

12


### Matrix Multiplication

```
'ik, kj -> ij'
```

| Axis | Meaning |
|------|---------|
| `i`  | Rows of A, rows of output |
| `j`  | Cols of B, cols of output |
| `k`  | Shared (contracted) axis — summed away |

This is the exact definition of matrix multiplication: output[i,j] = Σₖ A[i,k] · B[k,j].  
Equivalent to `A @ B` or `np.dot(A, B)`.


In [10]:
#Part 3: Matrix Multiplication
A = np.arange(6).reshape(2, 3)
B = np.arange(12).reshape(3, 4)

# NumPy way
np_result = A @ B   # or np.dot(A, B)

# einsum way
einsum_result = np.einsum('ik,kj->ij', A, B)

print(einsum_result)
# [[20 23 26 29]
#  [56 68 80 92]]

[[20 23 26 29]
 [56 68 80 92]]


### Batch Matrix Multiplication (3-D Tensors)

```
'bij, bjk -> bik'
```

- `b` = batch index (stays in output — not contracted)
- `j` = contracted axis within each matrix pair
- Each pair `(A[b], B[b])` is multiplied independently

Equivalent to `np.matmul(A, B)` when inputs are 3-D.  
This is extremely useful in deep learning where you process a *batch* of matrices simultaneously.


In [11]:
#Part 4: Batch Matrix Multiplication (3D Tensors)
batch_size = 10
A = np.random.rand(batch_size, 2, 3)
B = np.random.rand(batch_size, 3, 4)

#Numpy way
result = np.matmul(A, B)

#einsum way
result_1 = np.einsum('bij,bjk->bik', A, B)

print(result_1.shape)

(10, 2, 4)


### Outer Product

```
'i, j -> ij'
```

- `i` comes only from `a`, `j` only from `b`
- No shared index → **no contraction**
- Every pair (aᵢ, bⱼ) is multiplied → a 2-D grid of products

Equivalent to `np.outer(a, b)`.


In [12]:
#Example: Outer Product
a = np.array([1, 2, 3])
b = np.array([4, 5])

result = np.einsum('i,j->ij', a, b)

print(result)

[[ 4  5]
 [ 8 10]
 [12 15]]


### Diagonal Extraction

```
'ii -> i'
```

- Same index repeated on the **same** operand → selects diagonal elements
- `i` appears in output → keep each diagonal element (don't sum)

Equivalent to `np.diag(M)`.


In [13]:
#Example: Diagonal Extraction as Vector
M = np.arange(9).reshape(3, 3)

result = np.einsum('ii->i', M)
print(result)

[0 4 8]


### Real Application: Pairwise Squared Distances

Computing the squared Euclidean distance between every pair of points from two sets X and Y:

```
||xᵢ - yⱼ||² = ||xᵢ||² + ||yⱼ||² - 2 xᵢ·yⱼ
```

**Step 1:** Squared norms with `'ij,ij->i'` — element-wise product then sum over coords.  
**Step 2:** Cross dot-products with `'ik,jk->ij'` — every point in X dotted with every point in Y.  
**Step 3:** Broadcast and combine.

This avoids an explicit double loop and is the foundation of **k-NN** and **kernel methods**.


In [17]:
import numpy as np

X = np.random.rand(5, 3)  # 5 points in 3D
Y = np.random.rand(4, 3)  # 4 points in 3D

# Squared distances: ||X_i - Y_j||^2 = sum_k (X_ik - Y_jk)^2
# Expand to: sum_k X_ik^2 + sum_k Y_jk^2 - 2 * sum_k X_ik Y_jk

X_sq = np.einsum('ij,ij->i', X, X)[:, np.newaxis]  # Shape: (5, 1)
Y_sq = np.einsum('ij,ij->i', Y, Y)[np.newaxis, :]  # Shape: (1, 4)
XY = np.einsum('ik,jk->ij', X, Y)                  # Shape: (5, 4)

dist_sq = X_sq + Y_sq - 2 * XY

### `np.einsum_path` — Optimising Multi-Operand Contractions

When you chain three or more operands, the *order* of contractions matters enormously for speed.

```python
np.einsum('ijk,kl,lm->ijm', A, B, C)
```

If A is (100,200,50), B is (50,300), C is (300,400):
- Contracting A·B first → intermediate shape (100,200,300) → then ·C = OK
- Contracting B·C first → intermediate shape (50,400) → tiny! → then A·(BC) = much better

`np.einsum_path(..., optimize='optimal')` finds the best order automatically.  
`optimize='greedy'` is faster to compute and usually finds a near-optimal path.

> Always use `optimize=True` (or `'optimal'`) for three or more operands.


In [18]:
A = np.random.rand(100, 200, 50)
B = np.random.rand(50, 300)
C = np.random.rand(300, 400)

path_info = np.einsum_path('ijk,kl,lm->ijm', A, B, C, optimize='optimal')
print(path_info[0])


result = np.einsum('ijk,kl,lm->ijm', A, B, C, optimize='optimal')

['einsum_path', (1, 2), (0, 1)]


### 📝 Practice — Einsum Operations

Given:
- `A` shape (4, 3, 2)
- `B` shape (2, 5)
- `C` shape (4, 3)
- `D` shape (3, 6)

Work through three exercises using einsum:

1. **3-D × 2-D contraction** — contract last axis of A with first axis of B → (4, 3, 5)
2. **Weighted sum** — contract A's first two axes against C → shape (2,)
3. **Batch matrix multiply** — C @ D using einsum → shape (4, 6)


In [19]:
#Task: Given the following arrays, use einsum to perform the specified operations.

In [20]:
A = np.random.rand(4, 3, 2)
B = np.random.rand(2, 5)
C = np.random.rand(4, 3)
D = np.random.rand(3, 6)

In [21]:
#Multiply A (4×3×2) with B (2×5) along the last axis of A and first axis of B, resulting in shape (4, 3, 5).
result = np.einsum('ijk,kl->ijl', A, B)
print(result.shape)

(4, 3, 5)


In [22]:
#compute the sum of the element‑wise product of A's first two dimensions for each element along the last axis? 
#Let's do: Compute sum_{i,j} A[i,j,k] * C[i,j] resulting in shape (2,).
#(Hint: Use indices 'ijk,ij->k'.)
result = np.einsum('ijk,ij->k', A, C)
print(result.shape)

(2,)


In [23]:
#Perform batch outer product: For each of the 4 batches, compute the outer product of the 3‑element 
#vector C[i] with the 6‑element vector from D? Actually C shape is (4,3) and D is (3,6). Compute C @ D using einsum to get (4,6).
result = np.einsum('ij, jk->ik', C, D)
print(result.shape)

(4, 6)


### 🗺️ Pairwise Distances Within a Single Point Cloud

This applies the same trick as the X-Y distance example, but for distances *within* a single set of 4 points.

The distance matrix `dist_sq[i, j]` gives the squared Euclidean distance between point `i` and point `j`.

Key einsum calls:
- `'id,id->i'` — squared norm of each point (sum of squared coordinates)
- `'id,jd->ij'` — dot product between every pair of points

The diagonal of `dist_sq` should be (approximately) 0 since a point has zero distance to itself.


In [24]:
import numpy as np

# 4 points in 3D space
C = np.random.rand(4, 3)

# 1. Squared norms: ||C_i||^2 and ||C_j||^2
# 'id,id->i' squares each element and sums across the 3D coordinates (axis 'd')
# This gives a 1D array of shape (4,) containing the squared magnitude of each point.
C_sq = np.einsum('id,id->i', C, C)

# 2. Dot products: C_i * C_j
# 'id,jd->ij' computes the dot product between every point 'i' and every point 'j'
# This results in a (4, 4) matrix.
C_dot = np.einsum('id,jd->ij', C, C)

# 3. Combine using broadcasting
# We reshape C_sq to (4, 1) and (1, 4) so they broadcast over the (4, 4) grid.
dist_sq = C_sq[:, np.newaxis] + C_sq[np.newaxis, :] - 2 * C_dot

print("Shape of distance matrix:", dist_sq.shape)
print("\nSquared Distance Matrix:\n", dist_sq)

Shape of distance matrix: (4, 4)

Squared Distance Matrix:
 [[0.         1.06986769 1.00507921 1.02454047]
 [1.06986769 0.         0.01273978 0.62374455]
 [1.00507921 0.01273978 0.         0.72956993]
 [1.02454047 0.62374455 0.72956993 0.        ]]


---

## 📖 Topic 5: `sliding_window_view` — Zero-Copy Windowed Views

`numpy.lib.stride_tricks.sliding_window_view(arr, window_shape)` creates a **view** that presents every contiguous window of a given shape overlapping across the array — **without copying any data**.

### How it works (stride magic)
Under the hood NumPy manipulates the `strides` tuple to make the same memory block appear as a higher-dimensional array:
- For a 1-D array of length N with window W → output shape `(N-W+1, W)`
- Both dimensions step by the original element stride — so adjacent windows *share* elements in memory
- Zero bytes are allocated for the window data itself

### ⚠️ Key gotcha: views are read-only
Modifying the original array *does* change the window view (they share memory).  
Attempting to write to the view raises `ValueError: assignment destination is read-only`.  
If you need a writable copy: `patches = sliding_window_view(arr, w).copy()`.

### Common use-cases
| Task | How |
|------|-----|
| Moving average / rolling stats | `.mean(axis=-1)` on the view |
| Image patch extraction (convolution prep) | 2-D window over 2-D image |
| Time-series feature engineering | 1-D window over a sequence |
| Non-overlapping patches | Slice with step equal to window size |


In [ ]:
#Day 5, Topic 5: Stride Tricks – sliding_window_view for Rolling Operations

### 1-D Sliding Window

For a length-10 array with window size 4:
- Output shape: `(7, 4)` — 7 possible starting positions
- `windows[0]` = elements 0-3, `windows[1]` = elements 1-4, etc.
- Modifying `a[1] = 999` instantly changes `windows` because they share memory.


In [31]:
#1D Sliding Window
from numpy.lib.stride_tricks import sliding_window_view

a = np.arange(10)
windows = sliding_window_view(a, window_shape=4)

print(windows.shape)
a[1] = 999
print(windows)

(7, 4)
[[  0 999   2   3]
 [999   2   3   4]
 [  2   3   4   5]
 [  3   4   5   6]
 [  4   5   6   7]
 [  5   6   7   8]
 [  6   7   8   9]]


### 2-D Sliding Window (Image Patches)

For a 5×5 image with a 3×3 window:
- Output shape: `(3, 3, 3, 3)` — (num_row_positions, num_col_positions, win_rows, win_cols)
- `patches[i, j]` gives the 3×3 patch whose top-left corner is at `(i, j)`
- Used in **convolution**, **template matching**, and **patch-based learning**


In [37]:
#2D Sliding Window (Image Patches)
b = np.arange(25).reshape(5, 5)

patches = sliding_window_view(b, window_shape=(3, 3))

print(patches.shape)

for i in range(patches.shape[0]):
    for j in range(patches.shape[1]):
        print(f"Patch ({i}, {j}):\n", patches[i, j])

(3, 3, 3, 3)
Patch (0, 0):
 [[ 0  1  2]
 [ 5  6  7]
 [10 11 12]]
Patch (0, 1):
 [[ 1  2  3]
 [ 6  7  8]
 [11 12 13]]
Patch (0, 2):
 [[ 2  3  4]
 [ 7  8  9]
 [12 13 14]]
Patch (1, 0):
 [[ 5  6  7]
 [10 11 12]
 [15 16 17]]
Patch (1, 1):
 [[ 6  7  8]
 [11 12 13]
 [16 17 18]]
Patch (1, 2):
 [[ 7  8  9]
 [12 13 14]
 [17 18 19]]
Patch (2, 0):
 [[10 11 12]
 [15 16 17]
 [20 21 22]]
Patch (2, 1):
 [[11 12 13]
 [16 17 18]
 [21 22 23]]
Patch (2, 2):
 [[12 13 14]
 [17 18 19]
 [22 23 24]]


### Specifying an Axis

By default `sliding_window_view` applies the window across all axes simultaneously.  
Pass `axis=` to restrict it to a specific axis:

```python
sliding_window_view(arr_2d, window_shape=3, axis=1)
```

This slides a length-3 window only along axis 1 (columns), leaving axis 0 (rows) unchanged.  
Useful for per-row rolling calculations on a 2-D table.


In [41]:
#Specifying Axes
arr_2d = np.arange(12).reshape(3, 4)

windows = sliding_window_view(arr_2d, window_shape=3, axis=1)
print(windows.shape)

(3, 2, 3)


### Multi-Dimensional Windows on Specific Axes

You can apply different window sizes to different axes simultaneously:

```python
sliding_window_view(arr_3d, window_shape=(2, 2), axis=(1, 2))
```

This slides a 2×2 window over axes 1 and 2, leaving axis 0 intact.  
The output has shape: `(original_axis0, axis1-1, axis2-1, 2, 2)`.


In [43]:
#Multi‑Dimensional Windows on Multi‑Axes
arr_3d = np.arange(60).reshape(3, 4, 5)

windows = sliding_window_view(arr_3d, window_shape=(2, 2), axis=(1, 2))
print(windows.shape)

(3, 3, 4, 2, 2)


### Moving Average (Rolling Mean)

Classic application: smooth a noisy time series.

```python
avg = sliding_window_view(arr, window)
monthly_avg = avg.mean(axis=1)
```

- Each row of `avg` is one window of `window` elements
- `.mean(axis=1)` computes the mean of each window
- No explicit loop needed — NumPy's C-level mean runs over all windows in one call

This is equivalent to `pd.Series(arr).rolling(window).mean()` but faster for raw arrays.


In [44]:
#Moving Average
arr = np.random.rand(1000)
window = 50

avg = sliding_window_view(arr, window)
monthly_avg = avg.mean(axis=1)

print(monthly_avg.shape)

(951,)


### Image Convolution (Mean Filter / Blur)

Extract all `k×k` patches from a 2-D image and apply a filter:

```python
patches = sliding_window_view(image, (k, k))   # shape: (H-k+1, W-k+1, k, k)
blurred = patches.mean(axis=(2, 3))            # mean over the k×k window dims
```

- `axis=(2, 3)` collapses the two window dimensions into a scalar per position
- This computes a **box blur** (uniform mean filter) over the entire image with no loops
- For a Gaussian blur, replace `.mean()` with `np.einsum` and a Gaussian kernel


In [46]:
#Image Convolution
images = np.random.randint(0, 256, (512, 512), dtype=np.uint8)
kernel_size = 3

patches = sliding_window_view(images, (kernel_size, kernel_size))
print(patches.shape)

#Filtering patches
patches_mean = patches.mean(axis=(2, 3))
print(patches_mean.shape)

(510, 510, 3, 3)
(510, 510)


### 📝 Practice — Image Patch Extraction & Blurring

Given a 1000×1000 float32 grayscale image:

1. **Extract all 5×5 overlapping patches** → shape should be `(996, 996, 5, 5)`
2. **Apply a mean blur** → compute `.mean(axis=(-2,-1))` → shape `(996, 996)`
3. **Non-overlapping patches** → slice `patches[::5, ::5]` to get patches with stride 5

> **Why 996?** A 5×5 window starting at position `i` occupies indices `i` to `i+4`. The last valid starting position is `1000-5 = 995`, giving 996 starting positions (0 through 995).


In [47]:
#Task: You have a 2D grayscale image of shape (1000, 1000) represented as a float32 array. Use sliding_window_view to:
image_2d = np.random.rand(1000, 1000).astype(np.float32)

In [48]:
#Extract all 5×5 patches with stride 1 (overlapping). What is the shape of the resulting view?
patch_size = 5
patches = sliding_window_view(image_2d, (patch_size, patch_size))
print(patches.shape)

(996, 996, 5, 5)


In [49]:
#Apply a 5×5 Gaussian blur approximation by replacing each pixel with the mean of its 5×5 neighborhood. 
#Use the patches to compute the blurred image without loops.
#Verify that the blurred image has shape (996, 996).
blurred_image = patches.mean(axis=(-2, -1))
print(blurred_image.shape)

(996, 996)


In [50]:
#Instead of stride 1, 
#extract non‑overlapping 5×5 patches (stride 5). Hint: First slice the array with step 5, then apply sliding_window_view with window size 1?
non_overlapping_image = patches[::5, ::5]
print(non_overlapping_image.shape)

(200, 200, 5, 5)
